In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
import os
import glob
import torch
import mne
import numpy as np
from mne_connectivity import spectral_connectivity_epochs

def build_adjacency_matrix(edges: np.ndarray, num_nodes: int = 19) -> np.ndarray:
    A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    row_idx, col_idx = np.tril_indices(num_nodes, k=-1)
    A[row_idx, col_idx] = edges
    A = A + A.T
    np.fill_diagonal(A, 1.0)
    return A

def precompute_stgcn_dataset(input_dir: str, output_dir: str, min_epochs: int = 40, max_epochs: int = 100):
    os.makedirs(output_dir, exist_ok=True)
    files = glob.glob(os.path.join(input_dir, '*_clean-epo.fif'))
    
    target_channels = [
        'Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 
        'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz'
    ]
    # STRICT BOUNDARY: Floor set to 2.0 Hz to fit 5 cycles within the 4-second epoch
    bands = [(2.0, 4.0), (4.0, 8.0), (8.0, 13.0), (13.0, 30.0)]
    sfreq = 128.0 # STRICT: Ensure this matches your actual dataset sampling rate
    
    valid_subjects = 0
    total_tensors_saved = 0
    
    for file in files:
        filename = os.path.basename(file).replace('_clean-epo.fif', '')
        epochs = mne.read_epochs(file, preload=True, verbose=False)
        
        # 1. MINIMUM BOUNDARY
        if len(epochs) < min_epochs:
            print(f"Dropping {filename}: Only {len(epochs)} epochs.")
            continue
            
        # 2. SPATIAL TOPOLOGY BOUNDARY
        if not all(ch in epochs.ch_names for ch in target_channels):
            print(f"Dropping {filename}: Missing target channels.")
            continue
            
        # 3. MAXIMUM BOUNDARY STRATEGY (The 100 Epoch Cap)
        if len(epochs) > max_epochs:
            # Set seed for reproducible downsampling
            np.random.seed(42) 
            # Randomly select 100 epochs without replacement
            selected_idx = np.random.choice(len(epochs), max_epochs, replace=False)
            # Sort to maintain chronological order of the seizure/brain states
            selected_idx.sort()
            # Truncate the MNE object instantly
            epochs = epochs[selected_idx]
            
        valid_subjects += 1
        epochs = epochs.pick(target_channels)
        data = epochs.get_data(copy=False) # Shape will now be at most (100, 19, T)
        
        # --- FEATURE EXTRACTION (Only runs on the capped dataset) ---
        
        # A. NODE FEATURES (X)
        X_all = np.zeros((data.shape[0], 19, 4, 512), dtype=np.float32)
        for b_idx, (fmin, fmax) in enumerate(bands):
            filtered = epochs.copy().filter(l_freq=fmin, h_freq=fmax, verbose=False).get_data(copy=False)
            T_actual = filtered.shape[2]
            if T_actual < 512:
                filtered = np.pad(filtered, ((0,0), (0,0), (0, 512 - T_actual)), mode='constant')
            elif T_actual > 512:
                filtered = filtered[:, :, :512]
            X_all[:, :, b_idx, :] = filtered

        # B. EDGE FEATURES (A)
        node_indices = np.tril_indices(19, k=-1)
        con = spectral_connectivity_epochs(
            data, method='wpli', sfreq=sfreq, indices=node_indices,
            fmin=[b[0] for b in bands], fmax=[b[1] for b in bands], 
            faverage=True, verbose=False
        )
        edges_data = con.get_data() 
        
        # C. DISK SERIALIZATION
        for epoch_idx in range(data.shape[0]):
            X_tensor = torch.tensor(X_all[epoch_idx], dtype=torch.float32)
            A_tensor = torch.zeros((4, 19, 19), dtype=torch.float32)
            for b_idx in range(4):
                A_tensor[b_idx] = torch.tensor(build_adjacency_matrix(edges_data[:, b_idx]))
                
            save_path = os.path.join(output_dir, f"{filename}_epoch{epoch_idx:04d}.pt")
            torch.save({'X': X_tensor, 'A': A_tensor}, save_path)
            total_tensors_saved += 1
            
    print(f"Precomputation complete. {valid_subjects} subjects processed. {total_tensors_saved} total .pt files generated.")

if __name__ == "__main__":
    precompute_stgcn_dataset(
        input_dir='/kaggle/input/maaed-cleaned-epochs-archive-new', 
        output_dir='/kaggle/working/stgcn_pt_data',
        min_epochs=40,
        max_epochs=150
    )

Dropping male_positive_raw_47: Only 16 epochs.
Dropping female_positive_raw_35: Only 37 epochs.
Dropping female_negative_raw_36: Only 23 epochs.
Dropping male_positive_raw_37: Only 10 epochs.
Dropping male_positive_raw_3: Only 16 epochs.
Precomputation complete. 207 subjects processed. 28695 total .pt files generated.


In [2]:
!pip install mne_connectivity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 79.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 76.1 MB/s eta 0:00:00


In [4]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict, Tuple

class PrecomputedSTGCNDataset(Dataset):
    """
    Loads pre-extracted X and A tensors. Groups files by subject and class 
    so the SubjectAwareSampler can enforce its balancing constraints.
    """
    def __init__(self, pt_directory: str, allowed_subjects: set = None):
        self.directory = pt_directory
        self.allowed_subjects = allowed_subjects
        self.files = sorted(glob.glob(os.path.join(self.directory, '*.pt')))
        
        self.class_map = {
            'female_negative': 0,
            'female_positive': 1,
            'male_negative': 2,
            'male_positive': 3
        }
        
        # Required structures for SubjectAwareSampler
        self.class_subject_indices: Dict[int, Dict[str, List[int]]] = {0: {}, 1: {}, 2: {}, 3: {}}
        self.flat_registry: List[Tuple[str, int]] = []
        
        self._build_registry()

    def _build_registry(self):
        global_idx = 0
        for filepath in self.files:
            basename = os.path.basename(filepath)
            
            # Extract class label
            label = next((idx for name, idx in self.class_map.items() if basename.startswith(name)), None)
            if label is None:
                continue
                
            # Extract subject ID (filename minus the _epochXXXX.pt part)
            # Example: "female_negative_sub01_epoch0001.pt" -> "female_negative_sub01"
            # Extract subject ID (filename minus the _epochXXXX.pt part)
            subject_id = basename.rsplit('_epoch', 1)[0]
            
            # STRICT BOUNDARY: Filter by allowed subjects for CV splitting
            if self.allowed_subjects is not None and subject_id not in self.allowed_subjects:
                continue
                
            if subject_id not in self.class_subject_indices[label]:
                self.class_subject_indices[label][subject_id] = []
                
            self.class_subject_indices[label][subject_id].append(global_idx)
            self.flat_registry.append((filepath, label))
            global_idx += 1
            
        valid_subjects = sum(len(subs) for subs in self.class_subject_indices.values())
        print(f"Registry complete: {valid_subjects} valid subjects, {len(self.flat_registry)} total epochs.")

    def __len__(self) -> int:
        return len(self.flat_registry)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        filepath, label = self.flat_registry[idx]
        
        data_dict = torch.load(filepath, map_location='cpu', weights_only=True)
        X = data_dict['X'] # [19, 4, 512]
        A = data_dict['A'] # [4, 19, 19]
        Y = torch.tensor(label, dtype=torch.long)
        
        return X, A, Y

In [18]:
# --- VERIFICATION ---
if __name__ == "__main__":
    # ASSUMPTION: You have already run precompute_stgcn_dataset()
    dataset = PrecomputedSTGCNDataset('/kaggle/working/stgcn_pt_data')
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    for X_batch, A_batch, Y_batch in dataloader:
        print(f"Node Features (X): {X_batch.shape}") # Expected: [32, 19, 4, 512]
        print(f"Adjacency (A): {A_batch.shape}")     # Expected: [32, 4, 19, 19]
        print(f"Labels (Y): {Y_batch.shape}")        # Expected: [32]
        break

Registry complete: 207 valid subjects, 20152 total epochs.
Node Features (X): torch.Size([32, 19, 4, 512])
Adjacency (A): torch.Size([32, 4, 19, 19])
Labels (Y): torch.Size([32])


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class STGCN_Block(nn.Module):
    """
    Spatio-Temporal Graph Convolutional Block tailored for Dynamic EEG Connectivity.
    Routes node features through multiple functional adjacency matrices (microstates).
    """
    def __init__(self, in_channels: int, out_channels: int, num_microstates: int = 4, t_kernel_size: int = 9):
        super().__init__()
        
        # The Spatial Conv expands features by observing them through all 'K' microstates
        self.spatial_conv = nn.Conv2d(num_microstates * in_channels, out_channels, kernel_size=(1, 1))
        
        # Temporal Convolution (1D slide across time)
        padding = ((t_kernel_size - 1) // 2, 0)
        self.temporal_conv = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=(1, t_kernel_size),
            padding=(0, padding[0])
        )
        self.batch_norm = nn.BatchNorm2d(out_channels)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Node features [Batch, Channels, Nodes, Time] -> [B, C, V, T]
            A (torch.Tensor): Dynamic Adjacency [Batch, Microstates, Nodes, Nodes] -> [B, K, V, V]
        Returns:
            torch.Tensor: Convolved features [B, out_channels, V, T]
        """
        B, C, V, T = x.shape
        K = A.shape[1]

        # 1. SPATIAL GRAPH CONVOLUTION (Multiview Topological Routing)
        # We use Einstein Summation to multiply the node features by all 4 adjacency matrices.
        # b: batch, k: microstate, c: channel, v: target node, w: source node, t: time
        x_spatial = torch.einsum('bkvw, bcwt -> bkcvt', A, x) # Result: [B, K, C, V, T]

        # Collapse the Microstate (K) and Channel (C) dimensions to mix features
        x_spatial = x_spatial.reshape(B, K * C, V, T)
        
        # Apply 1x1 Convolution to reduce/transform back to desired out_channels
        x_spatial = F.relu(self.spatial_conv(x_spatial)) # Result: [B, out_channels, V, T]

        # 2. TEMPORAL CONVOLUTION
        x_temporal = self.temporal_conv(x_spatial)
        x_temporal = self.batch_norm(x_temporal)

        return F.relu(x_temporal)

class EpilepsySTGCN(nn.Module):
    """
    Full ST-GCN Architecture for 4-Class Epilepsy Diagnosis.
    """
    def __init__(self, num_classes: int = 4):
        super().__init__()
        
        # Architecture depth can be scaled by adding more blocks
        self.block1 = STGCN_Block(in_channels=4, out_channels=16, num_microstates=4)
        self.block2 = STGCN_Block(in_channels=16, out_channels=32, num_microstates=4)
        self.block3 = STGCN_Block(in_channels=32, out_channels=64, num_microstates=4)
        
        # Classifier Readout
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        # 1. TENSOR ALIGNMENT
        # Dataloader gives x as [Batch, Nodes(19), Channels(4), Time(512)].
        # PyTorch Conv2d strictly requires [Batch, Channels, Nodes, Time].
        x = x.permute(0, 2, 1, 3) # Transforms to [B, 4, 19, 512]
        
        # 2. SPATIO-TEMPORAL EXTRACTION
        x = self.block1(x, A)
        x = self.block2(x, A)
        x = self.block3(x, A) # Result: [B, 64, 19, 512]
        
        # 3. GLOBAL AVERAGE POOLING
        # Collapse the Spatial (Nodes: dim 2) and Temporal (Time: dim 3) dimensions
        x = x.mean(dim=(2, 3)) # Result: [B, 64]
        
        # 4. CLASSIFICATION
        return self.fc(x)

# --- USAGE & I/O SHAPE VERIFICATION ---
if __name__ == "__main__":
    # Mocking the exact output you just verified from your dataloader
    mock_X = torch.randn(32, 19, 4, 512)
    mock_A = torch.randn(32, 4, 19, 19)
    
    # Initialize your model
    model = EpilepsySTGCN(num_classes=4)
    
    # Forward Pass
    logits = model(mock_X, mock_A)
    
    print(f"Input X: {mock_X.shape}")
    print(f"Input A: {mock_A.shape}")
    print(f"Output Logits: {logits.shape}") # Expected: [32, 4]

Input X: torch.Size([32, 19, 4, 512])
Input A: torch.Size([32, 4, 19, 19])
Output Logits: torch.Size([32, 4])


In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class STGCN_Block(nn.Module):
    """
    Spatio-Temporal Graph Convolutional Block tailored for Dynamic EEG Connectivity.
    Routes node features through multiple functional adjacency matrices (microstates).
    """
    def __init__(self, in_channels: int, out_channels: int, num_microstates: int = 4, t_kernel_size: int = 9):
        super().__init__()
        
        # The Spatial Conv expands features by observing them through all 'K' microstates
        self.spatial_conv = nn.Conv2d(num_microstates * in_channels, out_channels, kernel_size=(1, 1))
        
        # Temporal Convolution (1D slide across time)
        padding = ((t_kernel_size - 1) // 2, 0)
        self.temporal_conv = nn.Conv2d(
            out_channels, out_channels,
            kernel_size=(1, t_kernel_size),
            padding=(0, padding[0])
        )
        self.batch_norm = nn.BatchNorm2d(out_channels)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Node features [Batch, Channels, Nodes, Time] -> [B, C, V, T]
            A (torch.Tensor): Dynamic Adjacency [Batch, Microstates, Nodes, Nodes] -> [B, K, V, V]
        Returns:
            torch.Tensor: Convolved features [B, out_channels, V, T]
        """
        B, C, V, T = x.shape
        K = A.shape[1]

        # 1. SPATIAL GRAPH CONVOLUTION (Multiview Topological Routing)
        # We use Einstein Summation to multiply the node features by all 4 adjacency matrices.
        # b: batch, k: microstate, c: channel, v: target node, w: source node, t: time
        x_spatial = torch.einsum('bkvw, bcwt -> bkcvt', A, x) # Result: [B, K, C, V, T]

        # Collapse the Microstate (K) and Channel (C) dimensions to mix features
        x_spatial = x_spatial.reshape(B, K * C, V, T)
        
        # Apply 1x1 Convolution to reduce/transform back to desired out_channels
        x_spatial = F.relu(self.spatial_conv(x_spatial)) # Result: [B, out_channels, V, T]

        # 2. TEMPORAL CONVOLUTION
        x_temporal = self.temporal_conv(x_spatial)
        x_temporal = self.batch_norm(x_temporal)

        return F.relu(x_temporal)

class EpilepsySTGCN(nn.Module):
    """
    Full ST-GCN Architecture for 4-Class Epilepsy Diagnosis.
    """
    def __init__(self, num_classes: int = 4):
        super().__init__()
        
        # Architecture depth can be scaled by adding more blocks
        self.block1 = STGCN_Block(in_channels=4, out_channels=16, num_microstates=4)
        self.block2 = STGCN_Block(in_channels=16, out_channels=32, num_microstates=4)
        self.block3 = STGCN_Block(in_channels=32, out_channels=64, num_microstates=4)
        
        # Classifier Readout
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        # 1. TENSOR ALIGNMENT
        # Dataloader gives x as [Batch, Nodes(19), Channels(4), Time(512)].
        # PyTorch Conv2d strictly requires [Batch, Channels, Nodes, Time].
        x = x.permute(0, 2, 1, 3) # Transforms to [B, 4, 19, 512]
        
        # 2. SPATIO-TEMPORAL EXTRACTION
        x = self.block1(x, A)
        x = self.block2(x, A)
        x = self.block3(x, A) # Result: [B, 64, 19, 512]
        
        # 3. GLOBAL AVERAGE POOLING
        # Collapse the Spatial (Nodes: dim 2) and Temporal (Time: dim 3) dimensions
        x = x.mean(dim=(2, 3)) # Result: [B, 64]
        
        # 4. CLASSIFICATION
        return self.fc(x)

# --- USAGE & I/O SHAPE VERIFICATION ---
if __name__ == "__main__":
    # Mocking the exact output you just verified from your dataloader
    mock_X = torch.randn(32, 19, 4, 512)
    mock_A = torch.randn(32, 4, 19, 19)
    
    # Initialize your model
    model = EpilepsySTGCN(num_classes=4)
    
    # Forward Pass
    logits = model(mock_X, mock_A)
    
    print(f"Input X: {mock_X.shape}")
    print(f"Input A: {mock_A.shape}")
    print(f"Output Logits: {logits.shape}") # Expected: [32, 4]

Input X: torch.Size([32, 19, 4, 512])
Input A: torch.Size([32, 4, 19, 19])
Output Logits: torch.Size([32, 4])


In [22]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold

# ASSUMPTION: You have the modified PrecomputedSTGCNDataset, SubjectAwareSampler, and EpilepsySTGCN in scope.

def get_subject_labels(pt_directory: str) -> Tuple[np.ndarray, np.ndarray]:
    """Scans the directory to map each unique subject to their single class label."""
    import glob
    files = glob.glob(os.path.join(pt_directory, '*.pt'))
    subject_to_label = {}
    
    class_map = {'female_negative': 0, 'female_positive': 1, 'male_negative': 2, 'male_positive': 3}
    
    for f in files:
        basename = os.path.basename(f)
        label = next((idx for name, idx in class_map.items() if basename.startswith(name)), None)
        if label is None: continue
        
        subject_id = basename.rsplit('_epoch', 1)[0]
        subject_to_label[subject_id] = label
        
    subjects = np.array(list(subject_to_label.keys()))
    labels = np.array(list(subject_to_label.values()))
    return subjects, labels

def train_kfold_cv(data_dir: str, num_folds: int = 5, epochs_per_fold: int = 50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Strict Hardware Verification: Training on {device}")
    
    subjects, labels = get_subject_labels(data_dir)
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(subjects, labels)):
        print(f"\n{'='*20} INITIALIZING FOLD {fold+1}/{num_folds} {'='*20}")
        
        train_subjects = set(subjects[train_idx])
        val_subjects = set(subjects[val_idx])
        
        # 1. ISOLATED DATASETS
        train_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=train_subjects)
        val_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=val_subjects)
        
        # 2. STRICT BATCHING LOGIC
        # Train gets the balanced sampler to prevent gradient domination
        train_sampler = SubjectAwareSampler(train_dataset, subjects_per_class=3, epochs_per_subject=4)
        train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2, pin_memory=True)
        
        # Validation uses standard sequential loading (no sampler) because we just evaluate all samples
        val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
        
        # 3. ABSOLUTE STATE ISOLATION (Destroy and Rebuild)
        model = EpilepsySTGCN(num_classes=4).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        
        # 4. EPOCH LOOP
        for epoch in range(epochs_per_fold):
            # --- TRAIN ---
            model.train()
            train_loss = 0.0
            
            for X_batch, A_batch, Y_batch in train_loader:
                X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                
                optimizer.zero_grad()
                logits = model(X_batch, A_batch)
                loss = criterion(logits, Y_batch)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
                
            # --- VALIDATION ---
            model.eval()
            val_loss = 0.0
            correct = 0
            total = 0
            
            with torch.no_grad():
                for X_batch, A_batch, Y_batch in val_loader:
                    X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                    logits = model(X_batch, A_batch)
                    loss = criterion(logits, Y_batch)
                    
                    val_loss += loss.item()
                    predictions = torch.argmax(logits, dim=1)
                    correct += (predictions == Y_batch).sum().item()
                    total += Y_batch.size(0)
                    
            val_acc = (correct / total) * 100
            print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.2f}%")
            
        fold_metrics.append(val_acc)
        print(f"Fold {fold+1} Final Validation Accuracy: {val_acc:.2f}%")
        
    print(f"\n{'-'*50}\nFINAL CROSS-VALIDATION RESULTS: {np.mean(fold_metrics):.2f}% ± {np.std(fold_metrics):.2f}%")



In [23]:
if __name__ == "__main__":
    # STRICT BOUNDARY: Do not alter the indentation.
    data_directory = '/kaggle/working/stgcn_pt_data'
    
    # Pre-execution I/O assertion
    import os
    if not os.path.exists(data_directory):
        raise FileNotFoundError(f"Fatal: Directory '{data_directory}' not found. You must run Stage 1 first.")
        
    print("Initiating Subject-Independent Stratified 5-Fold Cross-Validation...")
    
    # Execute the training loop
    train_kfold_cv(
        data_dir=data_directory, 
        num_folds=5, 
        epochs_per_fold=50
    )

Initiating Subject-Independent Stratified 5-Fold Cross-Validation...
Strict Hardware Verification: Training on cpu

==================== INITIALIZING FOLD 1/5 ====================
Registry complete: 165 valid subjects, 16068 total epochs.
Registry complete: 42 valid subjects, 4084 total epochs.


NameError: name 'SubjectAwareSampler' is not defined

In [6]:
import os
import glob
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, Sampler, DataLoader
from sklearn.model_selection import StratifiedKFold
from typing import List, Dict, Tuple, Iterator

# ==========================================
# 1. DATASET & SAMPLER
# ==========================================
class PrecomputedSTGCNDataset(Dataset):
    def __init__(self, pt_directory: str, allowed_subjects: set = None):
        self.directory = pt_directory
        self.allowed_subjects = allowed_subjects
        self.files = sorted(glob.glob(os.path.join(self.directory, '*.pt')))
        
        self.class_map = {
            'female_negative': 0, 'female_positive': 1,
            'male_negative': 2, 'male_positive': 3
        }
        
        self.class_subject_indices: Dict[int, Dict[str, List[int]]] = {0: {}, 1: {}, 2: {}, 3: {}}
        self.flat_registry: List[Tuple[str, int]] = []
        self._build_registry()

    def _build_registry(self):
        global_idx = 0
        for filepath in self.files:
            basename = os.path.basename(filepath)
            label = next((idx for name, idx in self.class_map.items() if basename.startswith(name)), None)
            if label is None: continue
                
            subject_id = basename.rsplit('_epoch', 1)[0]
            
            # CV Split Filtering
            if self.allowed_subjects is not None and subject_id not in self.allowed_subjects:
                continue
                
            if subject_id not in self.class_subject_indices[label]:
                self.class_subject_indices[label][subject_id] = []
                
            self.class_subject_indices[label][subject_id].append(global_idx)
            self.flat_registry.append((filepath, label))
            global_idx += 1

    def __len__(self) -> int:
        return len(self.flat_registry)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        filepath, label = self.flat_registry[idx]
        data_dict = torch.load(filepath, map_location='cpu', weights_only=True)
        return data_dict['X'], data_dict['A'], torch.tensor(label, dtype=torch.long)

class SubjectAwareSampler(Sampler):
    def __init__(self, dataset: PrecomputedSTGCNDataset, subjects_per_class: int, epochs_per_subject: int):
        self.dataset = dataset
        self.subjects_per_class = subjects_per_class
        self.epochs_per_subject = epochs_per_subject
        
        for cls_idx, subjects in self.dataset.class_subject_indices.items():
            if len(subjects) < subjects_per_class:
                raise ValueError(f"Class {cls_idx} has only {len(subjects)} subjects. Requested {subjects_per_class}.")
                
    def __iter__(self) -> Iterator[List[int]]:
        available_data = copy.deepcopy(self.dataset.class_subject_indices)
        
        while True:
            batch_indices = []
            valid_batch = True
            for cls_idx in range(4):
                valid_subjects = [sub for sub, eps in available_data[cls_idx].items() if len(eps) >= self.epochs_per_subject]
                if len(valid_subjects) < self.subjects_per_class:
                    valid_batch = False
                    break
                    
                selected_subjects = np.random.choice(valid_subjects, self.subjects_per_class, replace=False)
                for sub_id in selected_subjects:
                    np.random.shuffle(available_data[cls_idx][sub_id])
                    selected_eps = available_data[cls_idx][sub_id][:self.epochs_per_subject]
                    available_data[cls_idx][sub_id] = available_data[cls_idx][sub_id][self.epochs_per_subject:]
                    batch_indices.extend(selected_eps)
                    
            if not valid_batch: break
            np.random.shuffle(batch_indices)
            yield batch_indices

    def __len__(self) -> int:
        min_epochs_total = min(
            sum(len(eps) for eps in subjects.values()) 
            for subjects in self.dataset.class_subject_indices.values()
        )
        return min_epochs_total // (self.subjects_per_class * self.epochs_per_subject)

# ==========================================
# 2. MODEL ARCHITECTURE
# ==========================================
class STGCN_Block(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, num_microstates: int = 4, t_kernel_size: int = 9):
        super().__init__()
        self.spatial_conv = nn.Conv2d(num_microstates * in_channels, out_channels, kernel_size=(1, 1))
        padding = ((t_kernel_size - 1) // 2, 0)
        self.temporal_conv = nn.Conv2d(out_channels, out_channels, kernel_size=(1, t_kernel_size), padding=(0, padding[0]))
        self.batch_norm = nn.BatchNorm2d(out_channels)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        B, C, V, T = x.shape
        K = A.shape[1]
        x_spatial = torch.einsum('bkvw, bcwt -> bkcvt', A, x)
        x_spatial = x_spatial.reshape(B, K * C, V, T)
        x_spatial = F.relu(self.spatial_conv(x_spatial))
        x_temporal = self.batch_norm(self.temporal_conv(x_spatial))
        return F.relu(x_temporal)

class EpilepsySTGCN(nn.Module):
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.block1 = STGCN_Block(in_channels=4, out_channels=16)
        self.block2 = STGCN_Block(in_channels=16, out_channels=32)
        self.block3 = STGCN_Block(in_channels=32, out_channels=64)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 1, 3) # [B, 4, 19, 512]
        x = self.block1(x, A)
        x = self.block2(x, A)
        x = self.block3(x, A)
        x = x.mean(dim=(2, 3)) # Global Average Pooling
        return self.fc(x)

# ==========================================
# 3. K-FOLD TRAINING LOOP
# ==========================================
def get_subject_labels(pt_directory: str) -> Tuple[np.ndarray, np.ndarray]:
    files = glob.glob(os.path.join(pt_directory, '*.pt'))
    subject_to_label = {}
    class_map = {'female_negative': 0, 'female_positive': 1, 'male_negative': 2, 'male_positive': 3}
    
    for f in files:
        basename = os.path.basename(f)
        label = next((idx for name, idx in class_map.items() if basename.startswith(name)), None)
        if label is None: continue
        subject_id = basename.rsplit('_epoch', 1)[0]
        subject_to_label[subject_id] = label
        
    return np.array(list(subject_to_label.keys())), np.array(list(subject_to_label.values()))

def train_kfold_cv(data_dir: str, num_folds: int = 5, epochs_per_fold: int = 50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Strict Hardware Verification: Training on {device}")
    
    subjects, labels = get_subject_labels(data_dir)
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(subjects, labels)):
        print(f"\n{'='*20} INITIALIZING FOLD {fold+1}/{num_folds} {'='*20}")
        
        train_subjects = set(subjects[train_idx])
        val_subjects = set(subjects[val_idx])
        
        train_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=train_subjects)
        val_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=val_subjects)
        
        # Batch Size = 3 subjects * 4 classes * 4 epochs = 48
        train_sampler = SubjectAwareSampler(train_dataset, subjects_per_class=3, epochs_per_subject=4)
        train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=48, shuffle=False, num_workers=2, pin_memory=True)
        
        model = EpilepsySTGCN(num_classes=4).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        
        for epoch in range(epochs_per_fold):
            model.train()
            train_loss = 0.0
            for X_batch, A_batch, Y_batch in train_loader:
                X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                optimizer.zero_grad()
                logits = model(X_batch, A_batch)
                loss = criterion(logits, Y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                
            model.eval()
            val_loss = 0.0
            correct, total = 0, 0
            with torch.no_grad():
                for X_batch, A_batch, Y_batch in val_loader:
                    X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                    logits = model(X_batch, A_batch)
                    loss = criterion(logits, Y_batch)
                    val_loss += loss.item()
                    predictions = torch.argmax(logits, dim=1)
                    correct += (predictions == Y_batch).sum().item()
                    total += Y_batch.size(0)
                    
            val_acc = (correct / total) * 100
            print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.2f}%")
            
        fold_metrics.append(val_acc)
        print(f"Fold {fold+1} Final Validation Accuracy: {val_acc:.2f}%")
        
    print(f"\n{'-'*50}\nFINAL CV RESULTS: {np.mean(fold_metrics):.2f}% ± {np.std(fold_metrics):.2f}%")

# ==========================================
# 4. EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    data_directory = '/kaggle/working/stgcn_pt_data'
    if not os.path.exists(data_directory):
        raise FileNotFoundError(f"Fatal: Directory '{data_directory}' not found.")
        
    print("Initiating Subject-Independent Stratified 5-Fold Cross-Validation...")
    train_kfold_cv(data_dir=data_directory, num_folds=5, epochs_per_fold=50)

Initiating Subject-Independent Stratified 5-Fold Cross-Validation...
Strict Hardware Verification: Training on cuda

==================== INITIALIZING FOLD 1/5 ====================
Epoch 001 | Train Loss: 1.2841 | Val Loss: 4.0106 | Val Acc: 30.88%
Epoch 002 | Train Loss: 1.1728 | Val Loss: 1.6677 | Val Acc: 29.29%
Epoch 003 | Train Loss: 1.0737 | Val Loss: 4.7230 | Val Acc: 28.04%
Epoch 004 | Train Loss: 0.9786 | Val Loss: 2.4097 | Val Acc: 28.04%
Epoch 005 | Train Loss: 0.9013 | Val Loss: 1.9074 | Val Acc: 35.37%
Epoch 006 | Train Loss: 0.8084 | Val Loss: 3.6522 | Val Acc: 34.17%
Epoch 007 | Train Loss: 0.7180 | Val Loss: 5.8085 | Val Acc: 18.28%
Epoch 008 | Train Loss: 0.6733 | Val Loss: 6.2524 | Val Acc: 28.04%
Epoch 009 | Train Loss: 0.6294 | Val Loss: 11.4705 | Val Acc: 31.73%
Epoch 010 | Train Loss: 0.5561 | Val Loss: 3.3980 | Val Acc: 34.17%
Epoch 011 | Train Loss: 0.4977 | Val Loss: 3.5738 | Val Acc: 30.49%
Epoch 012 | Train Loss: 0.4825 | Val Loss: 4.4162 | Val Acc: 21.97%
Ep

KeyboardInterrupt: 

In [7]:
import os
import glob
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, Sampler, DataLoader
from sklearn.model_selection import StratifiedKFold
from typing import List, Dict, Tuple, Iterator

# ==========================================
# 1. DATASET & SAMPLER
# ==========================================
class PrecomputedSTGCNDataset(Dataset):
    """
    Lightning-fast disk loader for precomputed X and A tensors.
    Implements subject-level filtering for strict Cross-Validation isolation.
    """
    def __init__(self, pt_directory: str, allowed_subjects: set = None):
        self.directory = pt_directory
        self.allowed_subjects = allowed_subjects
        self.files = sorted(glob.glob(os.path.join(self.directory, '*.pt')))
        
        self.class_map = {
            'female_negative': 0, 'female_positive': 1,
            'male_negative': 2, 'male_positive': 3
        }
        
        self.class_subject_indices: Dict[int, Dict[str, List[int]]] = {0: {}, 1: {}, 2: {}, 3: {}}
        self.flat_registry: List[Tuple[str, int]] = []
        self._build_registry()

    def _build_registry(self) -> None:
        global_idx = 0
        for filepath in self.files:
            basename = os.path.basename(filepath)
            label = next((idx for name, idx in self.class_map.items() if basename.startswith(name)), None)
            if label is None: continue
                
            subject_id = basename.rsplit('_epoch', 1)[0]
            
            # STRICT BOUNDARY: Filter by allowed subjects for CV splitting
            if self.allowed_subjects is not None and subject_id not in self.allowed_subjects:
                continue
                
            if subject_id not in self.class_subject_indices[label]:
                self.class_subject_indices[label][subject_id] = []
                
            self.class_subject_indices[label][subject_id].append(global_idx)
            self.flat_registry.append((filepath, label))
            global_idx += 1

    def __len__(self) -> int:
        return len(self.flat_registry)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        filepath, label = self.flat_registry[idx]
        data_dict = torch.load(filepath, map_location='cpu', weights_only=True)
        return data_dict['X'], data_dict['A'], torch.tensor(label, dtype=torch.long)

class SubjectAwareSampler(Sampler):
    """
    Prevents gradient domination by strictly balancing subjects and classes in every batch.
    """
    def __init__(self, dataset: PrecomputedSTGCNDataset, subjects_per_class: int, epochs_per_subject: int):
        self.dataset = dataset
        self.subjects_per_class = subjects_per_class
        self.epochs_per_subject = epochs_per_subject
        
        for cls_idx, subjects in self.dataset.class_subject_indices.items():
            if len(subjects) < subjects_per_class:
                raise ValueError(f"Class {cls_idx} has only {len(subjects)} subjects. Requested {subjects_per_class}.")
                
    def __iter__(self) -> Iterator[List[int]]:
        available_data = copy.deepcopy(self.dataset.class_subject_indices)
        
        while True:
            batch_indices = []
            valid_batch = True
            for cls_idx in range(4):
                valid_subjects = [sub for sub, eps in available_data[cls_idx].items() if len(eps) >= self.epochs_per_subject]
                if len(valid_subjects) < self.subjects_per_class:
                    valid_batch = False
                    break
                    
                selected_subjects = np.random.choice(valid_subjects, self.subjects_per_class, replace=False)
                for sub_id in selected_subjects:
                    np.random.shuffle(available_data[cls_idx][sub_id])
                    selected_eps = available_data[cls_idx][sub_id][:self.epochs_per_subject]
                    available_data[cls_idx][sub_id] = available_data[cls_idx][sub_id][self.epochs_per_subject:]
                    batch_indices.extend(selected_eps)
                    
            if not valid_batch: break
            np.random.shuffle(batch_indices)
            yield batch_indices

    def __len__(self) -> int:
        min_epochs_total = min(
            sum(len(eps) for eps in subjects.values()) 
            for subjects in self.dataset.class_subject_indices.values()
        )
        return min_epochs_total // (self.subjects_per_class * self.epochs_per_subject)

# ==========================================
# 2. MODEL ARCHITECTURE
# ==========================================
class STGCN_Block(nn.Module):
    """
    Spatio-Temporal Graph Convolutional Block.
    Performs Multi-View Topological Routing using Einstein Summation.
    """
    def __init__(self, in_channels: int, out_channels: int, num_microstates: int = 4, t_kernel_size: int = 9):
        super().__init__()
        self.spatial_conv = nn.Conv2d(num_microstates * in_channels, out_channels, kernel_size=(1, 1))
        padding = ((t_kernel_size - 1) // 2, 0)
        self.temporal_conv = nn.Conv2d(out_channels, out_channels, kernel_size=(1, t_kernel_size), padding=(0, padding[0]))
        self.batch_norm = nn.BatchNorm2d(out_channels)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        B, C, V, T = x.shape
        K = A.shape[1]
        
        # Spatial Graph Convolutions across K dynamic microstates
        x_spatial = torch.einsum('bkvw, bcwt -> bkcvt', A, x)
        x_spatial = x_spatial.reshape(B, K * C, V, T)
        x_spatial = F.relu(self.spatial_conv(x_spatial))
        
        # Temporal Convolution
        x_temporal = self.batch_norm(self.temporal_conv(x_spatial))
        return F.relu(x_temporal)

class EpilepsySTGCN(nn.Module):
    """
    Full ST-GCN Architecture for Subject-Independent 4-Class Epilepsy Diagnosis.
    Includes Instance Normalization and Dropout to combat subject memorization.
    """
    def __init__(self, num_classes: int = 4):
        super().__init__()
        
        # CRITICAL FIX 1: Eradicate subject-specific amplitude bias per frequency band
        self.subject_norm = nn.InstanceNorm2d(4, affine=True)
        
        self.block1 = STGCN_Block(in_channels=4, out_channels=16)
        self.block2 = STGCN_Block(in_channels=16, out_channels=32)
        self.block3 = STGCN_Block(in_channels=32, out_channels=64)
        
        # CRITICAL FIX 2: Penalize feature co-adaptation
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        # Align: [Batch, Nodes(19), Channels(4), Time(512)] -> [B, 4, 19, 512]
        x = x.permute(0, 2, 1, 3) 
        
        # Normalize continuous voltage to zero-mean, unit-variance dynamically
        x = self.subject_norm(x)
        
        x = self.block1(x, A)
        x = self.block2(x, A)
        x = self.block3(x, A)
        
        # Global Average Pooling
        x = x.mean(dim=(2, 3)) 
        
        # Regularize and Classify
        x = self.dropout(x)
        return self.fc(x)

# ==========================================
# 3. K-FOLD TRAINING LOOP
# ==========================================
def get_subject_labels(pt_directory: str) -> Tuple[np.ndarray, np.ndarray]:
    files = glob.glob(os.path.join(pt_directory, '*.pt'))
    subject_to_label = {}
    class_map = {'female_negative': 0, 'female_positive': 1, 'male_negative': 2, 'male_positive': 3}
    
    for f in files:
        basename = os.path.basename(f)
        label = next((idx for name, idx in class_map.items() if basename.startswith(name)), None)
        if label is None: continue
        subject_id = basename.rsplit('_epoch', 1)[0]
        subject_to_label[subject_id] = label
        
    return np.array(list(subject_to_label.keys())), np.array(list(subject_to_label.values()))

def train_kfold_cv(data_dir: str, num_folds: int = 5, epochs_per_fold: int = 50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Strict Hardware Verification: Training on {device}")
    
    subjects, labels = get_subject_labels(data_dir)
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(subjects, labels)):
        print(f"\n{'='*20} INITIALIZING FOLD {fold+1}/{num_folds} {'='*20}")
        
        train_subjects = set(subjects[train_idx])
        val_subjects = set(subjects[val_idx])
        
        train_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=train_subjects)
        val_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=val_subjects)
        
        # Batch Size = 3 subjects * 4 classes * 4 epochs = 48
        train_sampler = SubjectAwareSampler(train_dataset, subjects_per_class=3, epochs_per_subject=4)
        train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=48, shuffle=False, num_workers=2, pin_memory=True)
        
        model = EpilepsySTGCN(num_classes=4).to(device)
        criterion = nn.CrossEntropyLoss()
        
        # CRITICAL FIX 3: Increased weight_decay from 1e-4 to 1e-2 to enforce extreme L2 regularization
        optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-2)
        
        for epoch in range(epochs_per_fold):
            model.train()
            train_loss = 0.0
            for X_batch, A_batch, Y_batch in train_loader:
                X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                optimizer.zero_grad()
                logits = model(X_batch, A_batch)
                loss = criterion(logits, Y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                
            model.eval()
            val_loss = 0.0
            correct, total = 0, 0
            with torch.no_grad():
                for X_batch, A_batch, Y_batch in val_loader:
                    X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                    logits = model(X_batch, A_batch)
                    loss = criterion(logits, Y_batch)
                    val_loss += loss.item()
                    predictions = torch.argmax(logits, dim=1)
                    correct += (predictions == Y_batch).sum().item()
                    total += Y_batch.size(0)
                    
            val_acc = (correct / total) * 100
            print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.2f}%")
            
        fold_metrics.append(val_acc)
        print(f"Fold {fold+1} Final Validation Accuracy: {val_acc:.2f}%")
        
    print(f"\n{'-'*50}\nFINAL CV RESULTS: {np.mean(fold_metrics):.2f}% ± {np.std(fold_metrics):.2f}%")

# ==========================================
# 4. EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    data_directory = '/kaggle/working/stgcn_pt_data'
    if not os.path.exists(data_directory):
        raise FileNotFoundError(f"Fatal: Directory '{data_directory}' not found.")
        
    print("Initiating Subject-Independent Stratified 5-Fold Cross-Validation...")
    train_kfold_cv(data_dir=data_directory, num_folds=5, epochs_per_fold=50)

Initiating Subject-Independent Stratified 5-Fold Cross-Validation...
Strict Hardware Verification: Training on cuda

==================== INITIALIZING FOLD 1/5 ====================
Epoch 001 | Train Loss: 1.2575 | Val Loss: 1.5721 | Val Acc: 25.14%
Epoch 002 | Train Loss: 1.1839 | Val Loss: 2.0455 | Val Acc: 19.53%
Epoch 003 | Train Loss: 1.1513 | Val Loss: 1.5865 | Val Acc: 24.75%
Epoch 004 | Train Loss: 1.1120 | Val Loss: 1.6822 | Val Acc: 28.53%
Epoch 005 | Train Loss: 1.0935 | Val Loss: 2.0029 | Val Acc: 23.31%
Epoch 006 | Train Loss: 1.0707 | Val Loss: 1.8574 | Val Acc: 17.72%
Epoch 007 | Train Loss: 1.0251 | Val Loss: 2.3821 | Val Acc: 22.75%
Epoch 008 | Train Loss: 0.9913 | Val Loss: 2.3535 | Val Acc: 26.24%
Epoch 009 | Train Loss: 0.9868 | Val Loss: 2.3026 | Val Acc: 32.05%
Epoch 010 | Train Loss: 0.9777 | Val Loss: 2.0709 | Val Acc: 29.09%
Epoch 011 | Train Loss: 0.8736 | Val Loss: 1.7324 | Val Acc: 39.76%
Epoch 012 | Train Loss: 0.8841 | Val Loss: 1.9932 | Val Acc: 27.26%
Epo

KeyboardInterrupt: 

In [5]:
import os
import glob
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, Sampler, DataLoader
from sklearn.model_selection import StratifiedKFold
from typing import List, Dict, Tuple, Iterator

# ==========================================
# 1. DATASET & SAMPLER (UPDATED FOR BINARY)
# ==========================================
class PrecomputedSTGCNDataset(Dataset):
    """
    Lightning-fast disk loader for precomputed X and A tensors.
    Remaps 4 demographic classes into 2 pathological classes (Binary).
    """
    def __init__(self, pt_directory: str, allowed_subjects: set = None):
        self.directory = pt_directory
        self.allowed_subjects = allowed_subjects
        self.files = sorted(glob.glob(os.path.join(self.directory, '*.pt')))
        
        # CRITICAL FIX 1: Binary Remapping
        # 0 = Non-Epileptic (Interictal), 1 = Epileptic (Preictal)
        self.class_map = {
            'female_negative': 0, 'male_negative': 0, 
            'female_positive': 1, 'male_positive': 1
        }
        
        self.class_subject_indices: Dict[int, Dict[str, List[int]]] = {0: {}, 1: {}}
        self.flat_registry: List[Tuple[str, int]] = []
        self._build_registry()

    def _build_registry(self) -> None:
        global_idx = 0
        for filepath in self.files:
            basename = os.path.basename(filepath)
            label = next((idx for name, idx in self.class_map.items() if basename.startswith(name)), None)
            if label is None: continue
                
            subject_id = basename.rsplit('_epoch', 1)[0]
            
            if self.allowed_subjects is not None and subject_id not in self.allowed_subjects:
                continue
                
            if subject_id not in self.class_subject_indices[label]:
                self.class_subject_indices[label][subject_id] = []
                
            self.class_subject_indices[label][subject_id].append(global_idx)
            self.flat_registry.append((filepath, label))
            global_idx += 1

    def __len__(self) -> int:
        return len(self.flat_registry)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        filepath, label = self.flat_registry[idx]
        data_dict = torch.load(filepath, map_location='cpu', weights_only=True)
        return data_dict['X'], data_dict['A'], torch.tensor(label, dtype=torch.long)

class SubjectAwareSampler(Sampler):
    """
    Prevents gradient domination by strictly balancing subjects and classes (Binary).
    """
    def __init__(self, dataset: PrecomputedSTGCNDataset, subjects_per_class: int, epochs_per_subject: int):
        self.dataset = dataset
        self.subjects_per_class = subjects_per_class
        self.epochs_per_subject = epochs_per_subject
        
        for cls_idx in range(2):
            subjects = self.dataset.class_subject_indices[cls_idx]
            if len(subjects) < subjects_per_class:
                raise ValueError(f"Class {cls_idx} has only {len(subjects)} subjects. Requested {subjects_per_class}.")
                
    def __iter__(self) -> Iterator[List[int]]:
        available_data = copy.deepcopy(self.dataset.class_subject_indices)
        
        while True:
            batch_indices = []
            valid_batch = True
            for cls_idx in range(2): # CRITICAL FIX 2: Loop only over 2 classes
                valid_subjects = [sub for sub, eps in available_data[cls_idx].items() if len(eps) >= self.epochs_per_subject]
                if len(valid_subjects) < self.subjects_per_class:
                    valid_batch = False
                    break
                    
                selected_subjects = np.random.choice(valid_subjects, self.subjects_per_class, replace=False)
                for sub_id in selected_subjects:
                    np.random.shuffle(available_data[cls_idx][sub_id])
                    selected_eps = available_data[cls_idx][sub_id][:self.epochs_per_subject]
                    available_data[cls_idx][sub_id] = available_data[cls_idx][sub_id][self.epochs_per_subject:]
                    batch_indices.extend(selected_eps)
                    
            if not valid_batch: break
            np.random.shuffle(batch_indices)
            yield batch_indices

    def __len__(self) -> int:
        min_epochs_total = min(
            sum(len(eps) for eps in subjects.values()) 
            for cls_idx, subjects in self.dataset.class_subject_indices.items()
        )
        return min_epochs_total // (self.subjects_per_class * self.epochs_per_subject)

# ==========================================
# 2. MODEL ARCHITECTURE (WITH ADVANCED FIXES)
# ==========================================
class STGCN_Block(nn.Module):
    """
    Spatio-Temporal Graph Convolutional Block.
    Upgraded with Learnable Edge Masks and Strided Temporal Convolutions.
    """
    def __init__(self, in_channels: int, out_channels: int, num_microstates: int = 4, t_kernel_size: int = 31, t_stride: int = 1):
        super().__init__()
        
        # FIX 3: Learnable Edge Mask to prevent Graph Oversmoothing
        self.edge_importance = nn.Parameter(torch.ones(num_microstates, 19, 19))
        
        self.spatial_conv = nn.Conv2d(num_microstates * in_channels, out_channels, kernel_size=(1, 1))
        
        padding = ((t_kernel_size - 1) // 2, 0)
        self.temporal_conv = nn.Conv2d(
            out_channels, out_channels, 
            kernel_size=(1, t_kernel_size), 
            stride=(1, t_stride), 
            padding=(0, padding[0])
        )
        self.batch_norm = nn.BatchNorm2d(out_channels)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        B, C, V, T = x.shape
        K = A.shape[1]
        
        # Sparsify the dense wPLI graph dynamically
        A_masked = A * self.edge_importance
        
        x_spatial = torch.einsum('bkvw, bcwt -> bkcvt', A_masked, x)
        x_spatial = x_spatial.reshape(B, K * C, V, T)
        x_spatial = F.relu(self.spatial_conv(x_spatial))
        
        x_temporal = self.batch_norm(self.temporal_conv(x_spatial))
        return F.relu(x_temporal)

class EpilepsySTGCN(nn.Module):
    """
    Full ST-GCN Architecture for Subject-Independent Binary Epilepsy Diagnosis.
    Upgraded with Spike-Preserving Max Pooling.
    """
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.subject_norm = nn.InstanceNorm2d(4, affine=True)
        
        # FIX 4: Expanded Temporal Receptive Fields with Downsampling
        self.block1 = STGCN_Block(in_channels=4, out_channels=16, t_kernel_size=31, t_stride=2)
        self.block2 = STGCN_Block(in_channels=16, out_channels=32, t_kernel_size=15, t_stride=2)
        self.block3 = STGCN_Block(in_channels=32, out_channels=64, t_kernel_size=7, t_stride=2)
        
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 1, 3) 
        x = self.subject_norm(x)
        
        x = self.block1(x, A)
        x = self.block2(x, A)
        x = self.block3(x, A)
        
        # FIX 5: Spike-Preserving Temporal Max Pooling
        x = x.amax(dim=3) 
        x = x.mean(dim=2) 
        
        x = self.dropout(x)
        return self.fc(x)

# ==========================================
# 3. K-FOLD TRAINING LOOP (BINARY)
# ==========================================
def get_subject_labels(pt_directory: str) -> Tuple[np.ndarray, np.ndarray]:
    files = glob.glob(os.path.join(pt_directory, '*.pt'))
    subject_to_label = {}
    class_map = {'female_negative': 0, 'male_negative': 0, 'female_positive': 1, 'male_positive': 1}
    
    for f in files:
        basename = os.path.basename(f)
        label = next((idx for name, idx in class_map.items() if basename.startswith(name)), None)
        if label is None: continue
        subject_id = basename.rsplit('_epoch', 1)[0]
        subject_to_label[subject_id] = label
        
    return np.array(list(subject_to_label.keys())), np.array(list(subject_to_label.values()))

def train_kfold_cv(data_dir: str, num_folds: int = 5, epochs_per_fold: int = 50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Strict Hardware Verification: Training on {device}")
    
    subjects, labels = get_subject_labels(data_dir)
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(subjects, labels)):
        print(f"\n{'='*20} INITIALIZING FOLD {fold+1}/{num_folds} {'='*20}")
        
        train_subjects = set(subjects[train_idx])
        val_subjects = set(subjects[val_idx])
        
        train_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=train_subjects)
        val_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=val_subjects)
        
        # Batch Size = 6 subjects * 2 classes * 4 epochs = 48 (Maintained batch size)
        train_sampler = SubjectAwareSampler(train_dataset, subjects_per_class=6, epochs_per_subject=4)
        train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=48, shuffle=False, num_workers=2, pin_memory=True)
        
        model = EpilepsySTGCN(num_classes=2).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-2)
        
        for epoch in range(epochs_per_fold):
            model.train()
            train_loss = 0.0
            for X_batch, A_batch, Y_batch in train_loader:
                X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                optimizer.zero_grad()
                logits = model(X_batch, A_batch)
                loss = criterion(logits, Y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                
            model.eval()
            val_loss = 0.0
            correct, total = 0, 0
            with torch.no_grad():
                for X_batch, A_batch, Y_batch in val_loader:
                    X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                    logits = model(X_batch, A_batch)
                    loss = criterion(logits, Y_batch)
                    val_loss += loss.item()
                    predictions = torch.argmax(logits, dim=1)
                    correct += (predictions == Y_batch).sum().item()
                    total += Y_batch.size(0)
                    
            val_acc = (correct / total) * 100
            print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.2f}%")
            
        fold_metrics.append(val_acc)
        print(f"Fold {fold+1} Final Validation Accuracy: {val_acc:.2f}%")
        
    print(f"\n{'-'*50}\nFINAL CV RESULTS: {np.mean(fold_metrics):.2f}% ± {np.std(fold_metrics):.2f}%")

# ==========================================
# 4. EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    data_directory = '/kaggle/working/stgcn_pt_data'
    if not os.path.exists(data_directory):
        raise FileNotFoundError(f"Fatal: Directory '{data_directory}' not found.")
        
    print("Initiating Subject-Independent Stratified 5-Fold Cross-Validation (Binary)...")
    train_kfold_cv(data_dir=data_directory, num_folds=5, epochs_per_fold=50)

Initiating Subject-Independent Stratified 5-Fold Cross-Validation (Binary)...
Strict Hardware Verification: Training on cuda

==================== INITIALIZING FOLD 1/5 ====================
Epoch 001 | Train Loss: 0.6751 | Val Loss: 0.7120 | Val Acc: 53.11%
Epoch 002 | Train Loss: 0.6134 | Val Loss: 0.6984 | Val Acc: 55.55%
Epoch 003 | Train Loss: 0.5724 | Val Loss: 0.7742 | Val Acc: 44.22%
Epoch 004 | Train Loss: 0.3013 | Val Loss: 1.8610 | Val Acc: 54.14%
Epoch 005 | Train Loss: 0.0647 | Val Loss: 2.8065 | Val Acc: 52.36%
Epoch 006 | Train Loss: 0.0432 | Val Loss: 1.8377 | Val Acc: 59.72%
Epoch 007 | Train Loss: 0.0088 | Val Loss: 2.0003 | Val Acc: 60.83%
Epoch 008 | Train Loss: 0.0247 | Val Loss: 1.7031 | Val Acc: 58.55%
Epoch 009 | Train Loss: 0.0154 | Val Loss: 3.0297 | Val Acc: 47.64%
Epoch 010 | Train Loss: 0.0249 | Val Loss: 1.9089 | Val Acc: 67.58%
Epoch 011 | Train Loss: 0.0202 | Val Loss: 2.1707 | Val Acc: 62.43%
Epoch 012 | Train Loss: 0.0047 | Val Loss: 1.9344 | Val Acc: 4

Exception in thread Thread-128 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 541, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource

KeyboardInterrupt: 

In [6]:
# ==========================================
# 2. MODEL ARCHITECTURE (WITH GRAPH NORMALIZATION)
# ==========================================
class STGCN_Block(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, num_microstates: int = 4, t_kernel_size: int = 31, t_stride: int = 1, dropout_rate: float = 0.3):
        super().__init__()
        self.spatial_conv = nn.Conv2d(num_microstates * in_channels, out_channels, kernel_size=(1, 1))
        
        padding = ((t_kernel_size - 1) // 2, 0)
        self.temporal_conv = nn.Conv2d(
            out_channels, out_channels, 
            kernel_size=(1, t_kernel_size), 
            stride=(1, t_stride), 
            padding=(0, padding[0])
        )
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.dropout = nn.Dropout2d(p=dropout_rate) # Spatial-Channel Dropout

    def normalize_A(self, A: torch.Tensor) -> torch.Tensor:
        """
        Calculates Symmetric Degree Normalization: D^{-1/2} A D^{-1/2}
        Prevents highly connected nodes from exploding the gradient.
        """
        B, K, V, _ = A.shape
        # Add self-loops (Identity matrix) to ensure nodes pass messages to themselves
        I = torch.eye(V, device=A.device).view(1, 1, V, V)
        A_tilde = A + I
        
        # Calculate Degree Matrix
        D = A_tilde.sum(dim=-1) # Shape: [B, K, V]
        D_inv_sqrt = torch.pow(D, -0.5)
        D_inv_sqrt[torch.isinf(D_inv_sqrt)] = 0.0 # Handle division by zero
        
        # Symmetrically normalize
        D_inv_sqrt_matrix = torch.diag_embed(D_inv_sqrt) # Shape: [B, K, V, V]
        A_norm = torch.matmul(torch.matmul(D_inv_sqrt_matrix, A_tilde), D_inv_sqrt_matrix)
        return A_norm

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        B, C, V, T = x.shape
        K = A.shape[1]
        
        # CRITICAL FIX 1: Normalize the graph to prevent hub-node dominance
        A_norm = self.normalize_A(A)
        
        # CRITICAL FIX 2: DropEdge during training to prevent subject fingerprinting
        if self.training:
            edge_mask = (torch.rand_like(A_norm) > 0.2).float() # Drop 20% of edges randomly
            A_norm = A_norm * edge_mask

        x_spatial = torch.einsum('bkvw, bcwt -> bkcvt', A_norm, x)
        x_spatial = x_spatial.reshape(B, K * C, V, T)
        x_spatial = F.relu(self.spatial_conv(x_spatial))
        
        # Block-wise regularization
        x_spatial = self.dropout(x_spatial)
        
        x_temporal = self.batch_norm(self.temporal_conv(x_spatial))
        return F.relu(x_temporal)

class EpilepsySTGCN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.subject_norm = nn.InstanceNorm2d(4, affine=True)
        
        # Progressive Spatio-Temporal Extraction
        self.block1 = STGCN_Block(in_channels=4, out_channels=16, t_kernel_size=31, t_stride=2, dropout_rate=0.2)
        self.block2 = STGCN_Block(in_channels=16, out_channels=32, t_kernel_size=15, t_stride=2, dropout_rate=0.3)
        self.block3 = STGCN_Block(in_channels=32, out_channels=64, t_kernel_size=7, t_stride=2, dropout_rate=0.4)
        
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 1, 3) 
        x = self.subject_norm(x)
        
        x = self.block1(x, A)
        x = self.block2(x, A)
        x = self.block3(x, A)
        
        x = x.amax(dim=3) # Temporal Max Pool (Spike Preserving)
        x = x.mean(dim=2) # Spatial Average Pool
        
        return self.fc(x)

# ==========================================
# 3. K-FOLD TRAINING LOOP (WITH SCHEDULER)
# ==========================================
def train_kfold_cv(data_dir: str, num_folds: int = 5, epochs_per_fold: int = 50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Strict Hardware Verification: Training on {device}")
    
    subjects, labels = get_subject_labels(data_dir)
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(subjects, labels)):
        print(f"\n{'='*20} INITIALIZING FOLD {fold+1}/{num_folds} {'='*20}")
        
        train_subjects = set(subjects[train_idx])
        val_subjects = set(subjects[val_idx])
        
        train_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=train_subjects)
        val_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=val_subjects)
        
        train_sampler = SubjectAwareSampler(train_dataset, subjects_per_class=6, epochs_per_subject=4)
        train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=48, shuffle=False, num_workers=2, pin_memory=True)
        
        model = EpilepsySTGCN(num_classes=2).to(device)
        criterion = nn.CrossEntropyLoss()
        
        # Lowered initial LR to prevent exploding validation loss
        optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-2)
        
        # CRITICAL FIX 3: Cosine Annealing to smoothly guide the optimizer into the global minimum
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_per_fold, eta_min=1e-6)
        
        best_val_acc = 0.0
        
        for epoch in range(epochs_per_fold):
            model.train()
            train_loss = 0.0
            for X_batch, A_batch, Y_batch in train_loader:
                X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                optimizer.zero_grad()
                logits = model(X_batch, A_batch)
                loss = criterion(logits, Y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                
            scheduler.step() # Step the learning rate
            
            model.eval()
            val_loss = 0.0
            correct, total = 0, 0
            with torch.no_grad():
                for X_batch, A_batch, Y_batch in val_loader:
                    X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                    logits = model(X_batch, A_batch)
                    loss = criterion(logits, Y_batch)
                    val_loss += loss.item()
                    predictions = torch.argmax(logits, dim=1)
                    correct += (predictions == Y_batch).sum().item()
                    total += Y_batch.size(0)
                    
            val_acc = (correct / total) * 100
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                
            print(f"Epoch {epoch+1:03d} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.2f}% | LR: {scheduler.get_last_lr()[0]:.2e}")
            
        fold_metrics.append(best_val_acc)
        print(f"Fold {fold+1} Best Validation Accuracy: {best_val_acc:.2f}%")
        
    print(f"\n{'-'*50}\nFINAL CV RESULTS (BEST EPOCHS): {np.mean(fold_metrics):.2f}% ± {np.std(fold_metrics):.2f}%")

In [7]:
# ==========================================
# 4. EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    # STRICT BOUNDARY: Ensure this path matches Stage 1 output exactly
    data_directory = '/kaggle/working/stgcn_pt_data'
    
    import os
    if not os.path.exists(data_directory):
        raise FileNotFoundError(f"Fatal: Directory '{data_directory}' not found. Execute Stage 1 feature extraction first.")
        
    print("Initiating Subject-Independent Stratified 5-Fold CV (Binary)...")
    print("Architecture: Degree Normalized FM-STGCN with DropEdge & Cosine Annealing")
    
    # Execute the training loop
    train_kfold_cv(
        data_dir=data_directory, 
        num_folds=5, 
        epochs_per_fold=50
    )

Initiating Subject-Independent Stratified 5-Fold CV (Binary)...
Architecture: Degree Normalized FM-STGCN with DropEdge & Cosine Annealing
Strict Hardware Verification: Training on cuda

==================== INITIALIZING FOLD 1/5 ====================
Epoch 001 | Train Loss: 0.6902 | Val Loss: 0.6934 | Val Acc: 49.84% | LR: 5.00e-04
Epoch 002 | Train Loss: 0.6856 | Val Loss: 0.6935 | Val Acc: 49.84% | LR: 4.98e-04
Epoch 003 | Train Loss: 0.6821 | Val Loss: 0.6931 | Val Acc: 50.16% | LR: 4.96e-04
Epoch 004 | Train Loss: 0.6830 | Val Loss: 0.6932 | Val Acc: 49.84% | LR: 4.92e-04
Epoch 005 | Train Loss: 0.6828 | Val Loss: 0.6931 | Val Acc: 50.16% | LR: 4.88e-04
Epoch 006 | Train Loss: 0.6812 | Val Loss: 0.6933 | Val Acc: 49.84% | LR: 4.82e-04
Epoch 007 | Train Loss: 0.6826 | Val Loss: 0.6931 | Val Acc: 49.84% | LR: 4.76e-04
Epoch 008 | Train Loss: 0.6825 | Val Loss: 0.6931 | Val Acc: 50.16% | LR: 4.69e-04
Epoch 009 | Train Loss: 0.6825 | Val Loss: 0.6931 | Val Acc: 50.16% | LR: 4.61e-04
Epo

KeyboardInterrupt: 

In [8]:
# ==========================================
# 2. MODEL ARCHITECTURE (REVIVED)
# ==========================================
class STGCN_Block(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, num_microstates: int = 4, t_kernel_size: int = 15):
        super().__init__()
        self.edge_importance = nn.Parameter(torch.ones(num_microstates, 19, 19))
        self.spatial_conv = nn.Conv2d(num_microstates * in_channels, out_channels, kernel_size=(1, 1))
        
        padding = ((t_kernel_size - 1) // 2, 0)
        self.temporal_conv = nn.Conv2d(
            out_channels, out_channels, 
            kernel_size=(1, t_kernel_size), 
            stride=(1, 1), # FIX 1: Removed aggressive stride to preserve temporal resolution
            padding=(0, padding[0])
        )
        self.batch_norm = nn.BatchNorm2d(out_channels)

    def normalize_A(self, A: torch.Tensor) -> torch.Tensor:
        B, K, V, _ = A.shape
        I = torch.eye(V, device=A.device).view(1, 1, V, V)
        A_tilde = A + I
        D = A_tilde.sum(dim=-1)
        D_inv_sqrt = torch.pow(D, -0.5)
        D_inv_sqrt[torch.isinf(D_inv_sqrt)] = 0.0
        D_inv_sqrt_matrix = torch.diag_embed(D_inv_sqrt)
        return torch.matmul(torch.matmul(D_inv_sqrt_matrix, A_tilde), D_inv_sqrt_matrix)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        B, C, V, T = x.shape
        K = A.shape[1]
        
        A_norm = self.normalize_A(A * self.edge_importance)
        
        # FIX 2: Removed DropEdge to restore signal integrity
        x_spatial = torch.einsum('bkvw, bcwt -> bkcvt', A_norm, x)
        x_spatial = x_spatial.reshape(B, K * C, V, T)
        x_spatial = F.relu(self.spatial_conv(x_spatial))
        
        x_temporal = self.batch_norm(self.temporal_conv(x_spatial))
        return F.relu(x_temporal)

class EpilepsySTGCN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.subject_norm = nn.InstanceNorm2d(4, affine=True)
        
        self.block1 = STGCN_Block(in_channels=4, out_channels=16, t_kernel_size=31)
        self.block2 = STGCN_Block(in_channels=16, out_channels=32, t_kernel_size=15)
        self.block3 = STGCN_Block(in_channels=32, out_channels=64, t_kernel_size=7)
        
        # FIX 3: Lowered global dropout to prevent gradient destruction
        self.dropout = nn.Dropout(p=0.3) 
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 1, 3) 
        x = self.subject_norm(x)
        
        x = self.block1(x, A)
        x = self.block2(x, A)
        x = self.block3(x, A)
        
        x = x.amax(dim=3) # Temporal Max Pool
        x = x.mean(dim=2) # Spatial Average Pool
        
        x = self.dropout(x)
        return self.fc(x)

In [9]:
# ==========================================
# 3. K-FOLD TRAINING LOOP (REACTIVE OPTIMIZER)
# ==========================================
def train_kfold_cv(data_dir: str, num_folds: int = 5, epochs_per_fold: int = 50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Strict Hardware Verification: Training on {device}")
    
    subjects, labels = get_subject_labels(data_dir)
    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(subjects, labels)):
        print(f"\n{'='*20} INITIALIZING FOLD {fold+1}/{num_folds} {'='*20}")
        
        train_subjects = set(subjects[train_idx])
        val_subjects = set(subjects[val_idx])
        
        train_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=train_subjects)
        val_dataset = PrecomputedSTGCNDataset(data_dir, allowed_subjects=val_subjects)
        
        train_sampler = SubjectAwareSampler(train_dataset, subjects_per_class=6, epochs_per_subject=4)
        train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=48, shuffle=False, num_workers=2, pin_memory=True)
        
        model = EpilepsySTGCN(num_classes=2).to(device)
        criterion = nn.CrossEntropyLoss()
        
        # FIX 4: Restored learning rate and drastically reduced L2 penalty
        optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        
        # FIX 5: Dynamic Scheduler - reduces LR only if validation loss gets stuck
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)
        
        best_val_acc = 0.0
        
        for epoch in range(epochs_per_fold):
            model.train()
            train_loss = 0.0
            for X_batch, A_batch, Y_batch in train_loader:
                X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                optimizer.zero_grad()
                logits = model(X_batch, A_batch)
                loss = criterion(logits, Y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                
            model.eval()
            val_loss = 0.0
            correct, total = 0, 0
            with torch.no_grad():
                for X_batch, A_batch, Y_batch in val_loader:
                    X_batch, A_batch, Y_batch = X_batch.to(device), A_batch.to(device), Y_batch.to(device)
                    logits = model(X_batch, A_batch)
                    loss = criterion(logits, Y_batch)
                    val_loss += loss.item()
                    predictions = torch.argmax(logits, dim=1)
                    correct += (predictions == Y_batch).sum().item()
                    total += Y_batch.size(0)
            
            avg_train_loss = train_loss/len(train_loader)
            avg_val_loss = val_loss/len(val_loader)
            val_acc = (correct / total) * 100
            
            # Step the dynamic scheduler based on validation loss
            scheduler.step(avg_val_loss)
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                
            print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}% | LR: {optimizer.param_groups[0]['lr']:.2e}")
            
        fold_metrics.append(best_val_acc)
        print(f"Fold {fold+1} Best Validation Accuracy: {best_val_acc:.2f}%")
        
    print(f"\n{'-'*50}\nFINAL CV RESULTS (BEST EPOCHS): {np.mean(fold_metrics):.2f}% ± {np.std(fold_metrics):.2f}%")

In [ ]:
# ==========================================
# 4. EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    # STRICT BOUNDARY: Ensure this path matches Stage 1 output exactly
    data_directory = '/kaggle/working/stgcn_pt_data'
    
    import os
    if not os.path.exists(data_directory):
        raise FileNotFoundError(f"Fatal: Directory '{data_directory}' not found. Execute Stage 1 feature extraction first.")
        
    print("Initiating Subject-Independent Stratified 5-Fold CV (Binary)...")
    print("Architecture: Degree Normalized FM-STGCN with DropEdge & Cosine Annealing")
    
    # Execute the training loop
    train_kfold_cv(
        data_dir=data_directory, 
        num_folds=5, 
        epochs_per_fold=50
    )

Initiating Subject-Independent Stratified 5-Fold CV (Binary)...
Architecture: Degree Normalized FM-STGCN with DropEdge & Cosine Annealing
Strict Hardware Verification: Training on cuda

==================== INITIALIZING FOLD 1/5 ====================
Epoch 001 | Train Loss: 0.7774 | Val Loss: 0.6906 | Val Acc: 59.52% | LR: 1.00e-03
Epoch 002 | Train Loss: 0.6356 | Val Loss: 0.6192 | Val Acc: 67.70% | LR: 1.00e-03
Epoch 003 | Train Loss: 0.6022 | Val Loss: 0.7283 | Val Acc: 47.89% | LR: 1.00e-03
Epoch 004 | Train Loss: 0.1983 | Val Loss: 2.7577 | Val Acc: 49.82% | LR: 1.00e-03
Epoch 005 | Train Loss: 0.0142 | Val Loss: 2.5400 | Val Acc: 65.40% | LR: 1.00e-03
Epoch 006 | Train Loss: 0.0018 | Val Loss: 1.8805 | Val Acc: 61.20% | LR: 1.00e-03
Epoch 007 | Train Loss: 0.0102 | Val Loss: 0.9874 | Val Acc: 55.28% | LR: 1.00e-03
Epoch 008 | Train Loss: 0.0024 | Val Loss: 3.0766 | Val Acc: 62.49% | LR: 5.00e-04
Epoch 009 | Train Loss: 0.0006 | Val Loss: 3.5744 | Val Acc: 61.18% | LR: 5.00e-04
Epo